<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Precipitation_ees_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt

# Capture the FacetGrid object returned by xarray's plotting function
g = pr_annual.precipitation.plot.contourf(
    x = 'lon',
    y = 'lat',
    col = 'time',
    col_wrap = 5,
    robust = True,
    levels = 20,
    cmap = "turbo_r",
    cbar_kwargs = {'orientation': 'horizontal', 'label': 'Precipitation', 'shrink': 0.5}
)

# Iterate through all axes in the FacetGrid and set the aspect ratio
for ax in g.axes.flat:
    ax.set_aspect('equal')

# Set a global title for the entire figure
plt.suptitle('Annual Precipitation Contour Plot', y=1.02) # Adjust y to prevent overlap with subplot titles

# Adjust layout to make space for suptitle and prevent overlap, and for the horizontal colorbar
plt.tight_layout(rect=[0.02, 0.05, 0.98, 0.95]) # Increased padding on all sides
plt.show()

In [ ]:
!pip install geopandas

Now that `geopandas` is installed, we can define a polygon to represent the Ilemi Triangle. For demonstration purposes, I will use approximate coordinates. You can replace these with more precise coordinates or load a GeoJSON file if you have one. We will then plot this polygon on each of the contourf plots.

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
m = geemap.Map()
m

In [ ]:
roi = m.draw_last_feature.geometry()
roi

In [ ]:
border = (
    ee.FeatureCollection("FAO/GAUL/2015/level1")
    .filterBounds(roi)
    .geometry().simplify(100)
)

m.addLayer(border, {}, "border")

In [ ]:
gpm = (
    ee.ImageCollection("UCSB-CHC/CHIRPS/V3/DAILY_RNL")
    .filterDate('2015-01-01','2025-12-31')
    .select('precipitation')
    .map(
        lambda img: img.clip(border).copyProperties(img, ['system:time_start'])
    )
)

gpm.getInfo()

In [ ]:
import xarray as xr

In [ ]:
pr = xr.open_dataset(
    gpm,
    engine = 'ee',
    crs = 'EPSG:4326',
    scale = 0.1,
    geometry = border
)


In [ ]:
pr

In [ ]:
pr = pr.sortby('time') * 730.001

In [ ]:
import numpy as np

In [ ]:
pr_annual = pr.resample(time = 'YE').sum('time')

In [ ]:
pr_annual = xr.where(pr_annual == 0,  np.nan, pr_annual)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
g = pr_annual.precipitation.plot.contourf(
    x = 'lon',
    y = 'lat',
    col = 'time',
    col_wrap = 5,
    robust = True,
    levels = 10,
    cmap = "turbo_r",
    cbar_kwargs = {'orientation': 'horizontal', 'label': 'Precipitation', 'shrink': 0.4}
)

for ax in g.axes.flat:
    ax.set_aspect('equal')

plt.suptitle('Annual Precipitation Contour Plot', y=1.02)

plt.tight_layout(rect=[0.02, 0.05, 0.98, 0.95])
plt.show()

In [ ]:
annual_change = pr_annual.diff(dim = 'time')
mean_change = annual_change.mean(dim = 'time')

mean_change.precipitation.plot.contourf(
    x = 'lon',
    y = 'lat',
    robust =True,
    levels = 10,
    cmap = "turbo_r"
)

plt.gca().set_aspect('equal')
plt.show()

In [ ]:
pr_annual_spatial_mean = pr_annual.mean(dim=['lat', 'lon'])

pr_annual_smoothed = pr_annual_spatial_mean.rolling(time=5, center=True).mean().fillna(pr_annual_spatial_mean)

plt.figure(figsize=(12, 6))

pr_annual_spatial_mean['precipitation'].plot(label='Original Annual Precipitation', color='blue', linestyle='-')

pr_annual_smoothed['precipitation'].plot(label='Smoothed Annual Precipitation (5-year rolling mean)', color='red', linestyle='--')

plt.title('Annual Mean Precipitation Time Series with Smoothing')
plt.xlabel('Year')
plt.ylabel('Mean Precipitation')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()